# Phase 5: Hybrid Inference + GSM8K Evaluation

**The critical test**. Compare 3 configs on GSM8K:
1. `base_only` — Qwen3.5-35B-A3B-Base with no intervention
2. `bias_only` — Add bias vector at L15 every token (paper's +13pp baseline)
3. `hybrid` — SAE classifies prompt's final L17 → apply matching category vector + bias at L15

**Expected (paper)**: bias_only +5-10pp, hybrid +8-15pp over base. With diffmeans vectors: maybe 50-70% of that.

**Budget**: ~2-3h compute on RTX 6000 Blackwell.

## Cell 1 — Setup (skip if same session as Phase 4b-alt)

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer', 'pandas', check=False)
    pip('uninstall', '-y', 'transformers', 'causal-conv1d', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, re, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/phase5'); OUT.mkdir(exist_ok=True)
HF_PHASE2 = 'caiovicentino1/qwen35-a3b-sae-phase2'
MODEL_ID = 'Qwen/Qwen3.5-35B-A3B-Base'
STEERING_LAYER = 15
SAE_LAYER = 17
N_LATENTS = 15
print('setup done')

## Cell 2 — Load Qwen3.5-35B-A3B-Base (skip if loaded)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download, HfApi

if 'model' not in dir() or 'tok' not in dir():
    tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tok.pad_token_id is None: tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
        attn_implementation='sdpa', trust_remote_code=True)
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
print(f'Model: {torch.cuda.memory_allocated()/1e9:.1f} GB')

def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

## Cell 3 — Load SAE + steering vectors + bias from HF

In [ ]:
path = snapshot_download(HF_PHASE2, repo_type='model', cache_dir='/content/cache')

# SAE for classification
class TopKSAE(nn.Module):
    def __init__(self, d_in, n, k=3):
        super().__init__()
        self.W_enc = nn.Parameter(torch.zeros(d_in, n))
        self.b_enc = nn.Parameter(torch.zeros(n))
        self.W_dec = nn.Parameter(torch.zeros(n, d_in))
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.k = k
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        return pre

sae_state = torch.load(str(Path(path) / f'sae_n{N_LATENTS}.pt'), map_location='cpu')
sae = TopKSAE(2048, N_LATENTS, k=3)
sae.load_state_dict(sae_state)
sae = sae.cuda().to(torch.bfloat16)
sae.eval()

def sae_classify(h):
    """h: [d_model] single activation → int cluster id (top-1)"""
    with torch.no_grad():
        pre = (h.unsqueeze(0) - sae.b_dec.unsqueeze(0)) @ sae.W_enc + sae.b_enc.unsqueeze(0)
    return pre.argmax(dim=-1).item()

# Load diffmeans vectors
vectors = {}
for cid in range(N_LATENTS):
    vp = Path(path) / f'steering_vectors_diffmeans/v_c{cid}.npz'
    if vp.exists():
        vectors[cid] = torch.from_numpy(np.load(vp)['vector'].astype(np.float32)).cuda().to(torch.bfloat16)
bias_vec = torch.from_numpy(np.load(Path(path) / 'steering_vectors_diffmeans/v_cbias.npz')['vector'].astype(np.float32)).cuda().to(torch.bfloat16)
print(f'Loaded {len(vectors)} category vectors + bias (norm {bias_vec.norm():.3f})')

## Cell 4 — Install hooks: L15 adder + L17 capturer

In [ ]:
# Remove any prior hooks
for var in ['h_capture', 'h_adder']:
    if var in dir():
        try: eval(var).remove()
        except: pass

# L17 capture (for SAE classification after forward)
l17_last = {'value': None}
def capture_hook(module, inp, out):
    h = out[0] if isinstance(out, tuple) else out
    # Capture LAST TOKEN only
    l17_last['value'] = h[:, -1, :].detach()  # [B, d]
h_capture = get_layer_module(model, SAE_LAYER).register_forward_hook(capture_hook)

# L15 adder (applies bias + optional category vector)
class L15Adder:
    def __init__(self, bias, vectors):
        self.bias = bias
        self.vectors = vectors
        self.mode = 'off'  # 'off' | 'bias_only' | 'hybrid'
        self.category = None  # set by hybrid
        self.alpha = 1.0
    def set(self, mode, category=None, alpha=1.0):
        self.mode = mode; self.category = category; self.alpha = alpha
    def make_hook(self):
        def hook(module, inp):
            if self.mode == 'off': return None
            h = inp[0]
            delta = self.bias * self.alpha
            if self.mode == 'hybrid' and self.category is not None and self.category in self.vectors:
                delta = delta + self.vectors[self.category] * self.alpha
            h = h + delta.to(h.dtype)
            return (h,) + inp[1:]
        return hook

adder = L15Adder(bias_vec, vectors)
h_adder = get_layer_module(model, STEERING_LAYER).register_forward_pre_hook(adder.make_hook())
print('✅ hooks installed: L17 capture + L15 adder')

## Cell 5 — Load GSM8K

In [ ]:
gsm_path = snapshot_download('openai/gsm8k', repo_type='dataset', cache_dir='/content/cache')
parquet_files = sorted(Path(gsm_path).rglob('*.parquet'))
test_files = [f for f in parquet_files if 'test' in f.name.lower()]
gsm = pd.read_parquet(test_files[0]).to_dict('records')
print(f'GSM8K test: {len(gsm)}')
print(f'Sample: {gsm[0]["question"][:200]}')
print(f'Answer: {gsm[0]["answer"][:200]}')

N_EVAL = 50
random.seed(42)
eval_sample = random.sample(gsm, N_EVAL)

def format_gsm(ex):
    return (f"Task: Answer the question below. Explain your reasoning step by step.\n\n"
            f"Question: {ex['question']}\n\nStep by step answer:")

GSM_ANSWER_RE = re.compile(r'####\s*([-\d\.,]+)')
ANSWER_EXTRACT_RE = re.compile(r'(?:answer|result|equals|=)\s*[:$]?\s*([-\d\.,]+)', re.IGNORECASE)

def extract_gold(answer_text):
    m = GSM_ANSWER_RE.search(answer_text)
    if m: return m.group(1).replace(',','').strip()
    return None

def extract_pred(text):
    # Try '####' marker first
    m = GSM_ANSWER_RE.search(text)
    if m: return m.group(1).replace(',','').strip()
    # Find last number
    nums = re.findall(r'-?\d+(?:\.\d+)?', text)
    return nums[-1] if nums else None

def is_correct(pred, gold):
    if pred is None or gold is None: return False
    try:
        return abs(float(pred) - float(gold)) < 1e-4
    except: return False

## Cell 6 — Eval 3 configs (~60-90 min)

In [ ]:
MAX_NEW = 256
CONFIGS = ['base_only', 'bias_only', 'hybrid']

def classify_prompt(prompt_ids):
    """Run one forward, capture L17 at last prompt token, classify via SAE."""
    adder.set('off')
    with torch.no_grad():
        _ = model(input_ids=prompt_ids, attention_mask=torch.ones_like(prompt_ids))
    h = l17_last['value'][0]  # [d]
    return sae_classify(h)

results = []
t0 = time.time()
for i, ex in enumerate(tqdm(eval_sample, desc='gsm8k eval')):
    try:
        gold = extract_gold(ex['answer'])
        if gold is None: continue
        p = format_gsm(ex)
        ids = tok(p, return_tensors='pt').input_ids.cuda()
        if ids.shape[1] > 2048: continue
        row = dict(idx=i, gold=gold)

        # Classify prompt once (for hybrid)
        category = classify_prompt(ids)
        row['category'] = category

        for cfg in CONFIGS:
            if cfg == 'base_only':
                adder.set('off')
            elif cfg == 'bias_only':
                adder.set('bias_only')
            elif cfg == 'hybrid':
                adder.set('hybrid', category=category)
            with torch.no_grad():
                out = model.generate(
                    input_ids=ids, attention_mask=torch.ones_like(ids),
                    max_new_tokens=MAX_NEW, do_sample=False,
                    pad_token_id=tok.pad_token_id, use_cache=True)
            resp = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
            pred = extract_pred(resp)
            row[f'{cfg}_pred'] = pred
            row[f'{cfg}_correct'] = is_correct(pred, gold)
            row[f'{cfg}_response'] = resp[:300]
        adder.set('off')
        results.append(row)
        if (i+1) % 5 == 0:
            accs = {c: sum(r[f'{c}_correct'] for r in results)/len(results) for c in CONFIGS}
            print(f'  [{i+1}/{N_EVAL}] ' + ' | '.join(f'{c}={a*100:.0f}%' for c,a in accs.items()))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue
    except Exception as e:
        print(f'q{i}: {e}'); continue

with open(OUT/'gsm8k_results.json', 'w') as f:
    json.dump(dict(n=len(results), configs=CONFIGS, results=results,
                   wall_min=(time.time()-t0)/60), f, indent=2, default=str)
print(f'\n✅ eval done in {(time.time()-t0)/60:.1f} min')

## Cell 7 — Verdict

In [ ]:
from scipy.stats import binomtest
from collections import Counter

print(f'=== GSM8K Hybrid Inference (N={len(results)}) ===\n')
base_acc = sum(r['base_only_correct'] for r in results) / len(results) * 100
print(f'base_only : {base_acc:.1f}%')
print(f'(paper Qwen2.5-32B+QwQ: base 92.6%, bias_only 89.9%, hybrid 94.4%)\n')

print(f'{"config":15s}  {"acc%":>6s}  {"Δpp":>6s}  {"gain":>5s}  {"lost":>5s}  {"p":>7s}')
bc = [r['base_only_correct'] for r in results]
for cfg in CONFIGS:
    if cfg == 'base_only': continue
    cc = [r[f'{cfg}_correct'] for r in results]
    acc = sum(cc) / len(results) * 100
    delta = acc - base_acc
    g = sum(1 for b,c in zip(bc,cc) if not b and c)
    l = sum(1 for b,c in zip(bc,cc) if b and not c)
    p = binomtest(g, g+l, p=0.5, alternative='two-sided').pvalue if g+l > 0 else 1.0
    star = ' ***' if delta>5 and l<g else (' **' if delta>2 and l<=g else (' *' if delta>0 else ''))
    print(f'{cfg:15s}  {acc:5.1f}%  {delta:+5.1f}  {g:>5d}  {l:>5d}  {p:>7.3f}{star}')

# Category usage
cats = Counter(r['category'] for r in results)
print(f'\nCategory usage (hybrid config): {dict(cats)}')

# Verdict
hybrid_acc = sum(r['hybrid_correct'] for r in results) / len(results) * 100
bias_acc = sum(r['bias_only_correct'] for r in results) / len(results) * 100
print('\n=== VERDICT ===')
if hybrid_acc >= base_acc + 5:
    print(f'*** BREAKTHROUGH: hybrid +{hybrid_acc-base_acc:.1f}pp > base')
    print(f'   Even with diffmeans vectors, SAE-steering pipeline works.')
    print(f'   Upgrade to trained vectors (vast.ai B200) for 91% gap recovery.')
elif hybrid_acc >= base_acc + 2:
    print(f'**  Moderate: hybrid +{hybrid_acc-base_acc:.1f}pp — diffmeans captures some signal')
elif bias_acc >= base_acc + 2:
    print(f'*   Bias alone helps (+{bias_acc-base_acc:.1f}pp), category routing doesn\'t add much')
else:
    print(f'-- Null: neither bias nor hybrid clearly beats base')
    print('   Pipeline issue OR diffmeans too weak. Consider trained vectors.')